# Book Recommender System

In [2]:
import numpy as np
import pandas as pd

In [3]:
books = pd.read_csv('data/books.csv', encoding='latin-1')

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20140\1563079984.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('data/books.csv', encoding='latin-1')


In [4]:
books.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [5]:
books.shape

(271360, 8)

In [6]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']]

In [7]:
books.rename(columns={'Book-Title': 'Title', 'Book-Author': 'Author', 'Year-Of-Publication': 'Year', 'Publisher': 'Publisher'}, inplace=True)

In [8]:
books.head(2)

,ISBN,Title,Author,Year,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada


In [9]:
users = pd.read_csv('data/users.csv')

In [10]:
users.head(2)

,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0


In [11]:
users.rename(columns={'User-Id': 'user_id', 'Location': 'location', 'Age': 'age'}, inplace=True)

In [12]:
ratings = pd.read_csv('data/ratings.csv')

In [13]:
ratings.head(2)

,User-ID,ISBN,Book-Rating
0,276725,034545104X,0
1,276726,0155061224,5


In [14]:
ratings.rename(columns={'User-ID': 'user_id', 'Book-Rating': 'book_rating'}, inplace=True)

In [15]:
ratings.columns

Index(['user_id', 'ISBN', 'book_rating'], dtype='object')

In [16]:
print(books.shape)
print(users.shape)
print(ratings.shape)

(271360, 5)
(278858, 3)
(1149780, 3)


### Finding each users with their total ratings on books

In [17]:
ratings['user_id'].value_counts()

user_id
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
116180        1
116166        1
116154        1
116137        1
276723        1
Name: count, Length: 105283, dtype: int64

value_counts() returns a series with index and value
considering only those users who rated more than 200 books

In [18]:
ratings['user_id'].value_counts()>200

user_id
11676      True
198711     True
153662     True
98391      True
35859      True
          ...  
116180    False
116166    False
116154    False
116137    False
276723    False
Name: count, Length: 105283, dtype: bool

Storing the returned series in variable x with user_id as index of the series and boolean value(true,false) as values

In [19]:
x = ratings['user_id'].value_counts()>200

In [20]:
x.shape

(105283,)

In [21]:
# keeping only the users who have rated more than 200 books
y = x[x].index

In [22]:
y.shape

(899,)

### Only 899 users are our real users. Because we will consider their ratings as valid ratings

In [23]:
# filtering the ratings dataframe to only include users who have rated more than 200 books
ratings = ratings[ratings['user_id'].isin(y)]

In [24]:
ratings.shape

(526356, 3)

In [25]:
ratings_with_books = ratings.merge(books, on='ISBN')

In [26]:
ratings_with_books.shape

(487671, 7)

In [27]:
ratings_with_books.head()

,user_id,ISBN,book_rating,Title,Author,Year,Publisher
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books


In [28]:
books.head()

,ISBN,Title,Author,Year,Publisher
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company


In [29]:
# counting the number of ratings for each book. It is a series with book titles as index and number of ratings as values
number_of_ratings = ratings_with_books.groupby('Title')['book_rating'].count()

In [30]:
number_of_ratings

Title
 A Light in the Storm: The Civil War Diary of Amelia Martin, Fenwick Island, Delaware, 1861 (Dear America)    2
 Always Have Popsicles                                                                                        1
 Apple Magic (The Collector's series)                                                                         1
 Beyond IBM: Leadership Marketing and Finance for the 1990s                                                   1
 Clifford Visita El Hospital (Clifford El Gran Perro Colorado)                                                1
                                                                                                             ..
Ã?Ã?ber die Pflicht zum Ungehorsam gegen den Staat.                                                         3
Ã?Ã?lpiraten.                                                                                               1
Ã?Ã?rger mit Produkt X. Roman.                                                                  

In [31]:
# resetting the index of the series to convert it back to a dataframe
number_of_ratings = number_of_ratings.reset_index()
number_of_ratings.head()

,Title,book_rating
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [32]:
number_of_ratings.rename(columns={'book_rating': 'number_of_ratings'}, inplace=True)

In [33]:
number_of_ratings.head(2)
number_of_ratings.shape

(160269, 2)

In [34]:
final_ratings = ratings_with_books.merge(number_of_ratings, on='Title')

In [35]:
final_ratings.shape

(487671, 8)

In [36]:
final_ratings.head()

,user_id,ISBN,book_rating,Title,Author,Year,Publisher,number_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,13


In [37]:
# keeping only those books which have more than 50 ratings
final_ratings = final_ratings[final_ratings['number_of_ratings'] > 50]

In [38]:
final_ratings.shape

(59903, 8)

In [ ]:
# check for duplicates in final_ratings 
final_ratings.duplicated().any() # there are no duplicates

False

In [ ]:
# creating a pivot table with book titles as index, user ids as columns and book ratings as values
book_pivot = final_ratings.pivot_table(columns='user_id', index='Title', values='book_rating')

In [41]:
book_pivot.shape

(703, 888)

### There are 703 books(reviewed more than 50 times by users) and 888 users(who reviewed more than 200 books)

In [42]:
book_pivot.head()

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
1st to Die: A Novel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2nd Chance,NaN,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4 Blondes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84 Charing Cross Road,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,10.0,NaN,NaN,NaN,NaN


In [43]:
book_pivot.fillna(0, inplace=True)

In [44]:
book_pivot.head(2)

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
Title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
from scipy.sparse import  csr_matrix
book_sparse = csr_matrix(book_pivot)

In [46]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm='brute')

In [47]:
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [60]:
# finding the 5 nearest neighbors for the first book in the pivot table
# values.reshape(1, -1)

#     Original: 1D array → shape like (features,)

#     After reshape: 2D array → shape (1, features)
# distances are distance between books in hyperspace & suggestions are the indices of the nearest neighbors
distances, suggestions = model.kneighbors(book_pivot.iloc[224,:].values.reshape(1,-1), n_neighbors=6) 

In [61]:
distances

array([[ 0.        , 67.73129098, 67.77802823, 72.22091879, 76.03909813,
        76.55027397]])

In [62]:
suggestions

array([[224, 227, 225, 228, 173, 277]], dtype=int64)

In [63]:
for i in range(len(suggestions)):
    print(book_pivot.index[suggestions[i]])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'Jacob Have I Loved'],
      dtype='object', name='Title')


In [55]:
book_pivot.index[54]

'At Home in Mitford (The Mitford Years)'

In [65]:
book_pivot.index[1]

'1st to Die: A Novel'

In [57]:
'Harry Potter and the Chamber of Secrets (Book 2)' in book_pivot.index

True

In [68]:
book_pivot.index.get_loc('Animal Farm')


47

In [73]:
def recommend_books(book_name):
    book_index = book_pivot.index.get_loc(book_name)
    distances, suggestions = model.kneighbors(book_pivot.iloc[book_index,:].values.reshape(1,-1), n_neighbors=6) 
    for i in range(len(suggestions[0])):
        if book_pivot.index[suggestions[0][i]] != book_name:
            print(book_pivot.index[suggestions[0][i]])

In [76]:
recommend_books('Animal Farm')

Exclusive
Jacob Have I Loved
Pleading Guilty
No Safe Place
Winter Moon
